# SETTING UP ACCESS TO GITHUB (PRESERVED STATE) FOR EVERY NEW RUNTIME

In [ ]:
from google.colab import userdata
import os

git_token = userdata.get('GIT_TOKEN')
git_username = "TalhaShoyo10"
repo_name = "CS-6304_PA0_28100131"
repo_url = f"https://{git_token}@github.com/{git_username}/{repo_name}.git"



if not os.path.exists(repo_name):
  !git clone {repo_url}
  print("Repo did not exist in local file system, cloned from github to initiate work on task.")
else:
  !git -C {repo_name} pull
  print("Repo already existed in local file system, pulled from github to catch up on any remote changes.")

Imports and Setup

In [ ]:
!pip install -q transformers

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
import json
import os
from transformers import ViTImageProcessor, ViTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# UNDERSTANDING ViT

Loading Pretrained ViT Model and Image Processor

In [ ]:
#Using a small pretrained ViT (base, patch size 16, 224 resolution), pretrained on ImageNet-21k and fine-tuned on ImageNet-1k
checkpoint = "google/vit-base-patch16-224"

processor = ViTImageProcessor.from_pretrained(checkpoint)
model = ViTForImageClassification.from_pretrained(checkpoint, output_attentions=True)
model = model.to(device)
model.eval()

print(f"Loaded checkpoint: {checkpoint}")
print(f"Patch size: {model.config.patch_size}, Image size: {model.config.image_size}")
print(f"Number of labels: {model.config.num_labels}")

## Image Classification with the Pretrained ViT

Selecting Test Images

In [ ]:
#Selecting a small set of test images from stable, publicly hosted URLs
#img_1: two cats on a couch (the standard sample used in the HuggingFace ViT documentation)
#img_2: a dog (standard sample used in the PyTorch Hub tutorials)
#img_3: left as an easy slot to swap in any image of choice (e.g. an unusual/ambiguous image) to probe the model further

image_urls = {
    "cats_on_couch": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "dog": "https://github.com/pytorch/hub/raw/master/images/dog.jpg",
}

images = {}
for name, url in image_urls.items():
    img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    images[name] = img

fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
if len(images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

Running Inference and Recording Top-1 Predictions

In [ ]:
predictions = {}

for name, img in images.items():
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    top1_prob, top1_idx = torch.max(probs, dim=-1)
    predicted_label = model.config.id2label[top1_idx.item()]

    predictions[name] = {
        "predicted_label": predicted_label,
        "confidence": top1_prob.item()
    }

    print(f"Image: {name}")
    print(f"Top-1 Prediction: {predicted_label} (confidence: {top1_prob.item():.4f})")
    print("")

## Visualizing Patch Attention

Extracting the [CLS] Token's Attention to Patch Tokens from the Final Layer

In [ ]:
def get_cls_attention_map(img, model, processor, device):
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    #outputs.attentions is a tuple of length num_layers, each of shape (batch, num_heads, seq_len, seq_len)
    final_layer_attn = outputs.attentions[-1]

    #Averaging over all attention heads in the final layer
    attn_avg_heads = final_layer_attn.mean(dim=1).squeeze(0)  # (seq_len, seq_len)

    #Row 0 corresponds to the [CLS] token; taking its attention to all patch tokens (excluding itself)
    cls_attention = attn_avg_heads[0, 1:]

    num_patches = cls_attention.shape[0]
    grid_size = int(num_patches ** 0.5)
    assert grid_size * grid_size == num_patches, "Number of patches is not a perfect square"

    attention_map = cls_attention.reshape(grid_size, grid_size).cpu().numpy()
    return attention_map, inputs

Overlaying the Attention Map on Each Image

In [ ]:
from scipy.ndimage import zoom

def overlay_attention(img, attention_map, alpha=0.5):
    #Resizing the low-resolution patch attention map up to the original image resolution
    img_resized = img.resize((224, 224))
    img_np = np.array(img_resized).astype(np.float32) / 255.0

    zoom_factor = 224 / attention_map.shape[0]
    attn_upsampled = zoom(attention_map, zoom_factor, order=1)
    attn_upsampled = (attn_upsampled - attn_upsampled.min()) / (attn_upsampled.max() - attn_upsampled.min() + 1e-8)

    return img_np, attn_upsampled

fig, axes = plt.subplots(len(images), 2, figsize=(10, 5 * len(images)))

attention_maps = {}
for row, (name, img) in enumerate(images.items()):
    attn_map, _ = get_cls_attention_map(img, model, processor, device)
    attention_maps[name] = attn_map
    img_np, attn_upsampled = overlay_attention(img, attn_map)

    ax_left = axes[row, 0] if len(images) > 1 else axes[0]
    ax_right = axes[row, 1] if len(images) > 1 else axes[1]

    ax_left.imshow(img_np)
    ax_left.set_title(f"{name} (predicted: {predictions[name]['predicted_label']})")
    ax_left.axis("off")

    ax_right.imshow(img_np)
    ax_right.imshow(attn_upsampled, cmap="jet", alpha=alpha)
    ax_right.set_title(f"{name} - CLS attention overlay")
    ax_right.axis("off")

plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task2_vit/attention_overlays.png", dpi=150)
plt.show()

## Analysis

Discussion points to fill in after reviewing the overlays and predictions above:

- Did the ViT's attention focus on image regions corresponding to the predicted object class, or did it attend to background / irrelevant regions?
- How does this attention-based explanation conceptually compare with CNN interpretation methods such as CAM or Grad-CAM (no implementation needed, discussion only)?
- Do different attention heads appear specialized (this can be probed further by visualizing per-head attention instead of the head-averaged map)?
- What advantage does a transformer's built-in attention mechanism offer for interpretability compared to post-hoc CNN saliency methods?

Saving Results

In [ ]:
vit_results = {
    "checkpoint": checkpoint,
    "predictions": predictions,
    "patch_grid_size": int(attention_maps[list(attention_maps.keys())[0]].shape[0]),
}

with open("CS-6304_PA0_28100131/results/task2_vit/vit_results.json", "w") as f:
    json.dump(vit_results, f, indent=2)

print("Results saved successfully !!")

Checking for Saved Files

In [ ]:
print(os.listdir("CS-6304_PA0_28100131/results/task2_vit"))

Git Configuration

In [ ]:
!git config --global user.email "thebenbat5@gmail.com"
!git config --global user.name "TalhaShoyo10"

Commiting work to Github

In [ ]:
commit_message = "Task 2 - ViT classification and attention visualization"
!git -C {repo_name} add -A
!git -C {repo_name} commit -m "{commit_message}"
!git -C {repo_name} push https://{git_token}@github.com/{git_username}/{repo_name}.git main